[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/tkowalski29/048785b387185428ca1f1912c615ad7b/flux1-dev-gradio.ipynb)

# 🎨 FLUX.1-dev Image Generator with Gradio

This notebook sets up FLUX.1-dev model with an interactive Gradio interface.

---

## 🔴 KROK 1: Najpierw włącz GPU! (wymagane)

**Google Colab wymaga ręcznego włączenia GPU. Zrób to PRZED uruchomieniem:**

**Kliknij w górnym menu:**
```
Runtime → Change runtime type → Hardware accelerator → GPU → Save
```

**⚠️ Bez GPU notebook nie zadziała!** System automatycznie sprawdzi to w pierwszej komórce.

---

## ✅ KROK 2: Uruchom notebook

Po włączeniu GPU:
1. Kliknij **Runtime** → **Run all**
2. **Przyznaj dostęp do Google Drive** (pojawi się popup - to potrzebne do zapisania modeli)
3. Notebook automatycznie:
   - Zainstaluje zależności (~2-3 min)
   - Pobierze modele (~10GB, 5-10 min) **- tylko pierwszy raz!**
   - Załaduje modele (~1 min)
   - Uruchomi Gradio
4. Kliknij w wygenerowany link Gradio
5. Generuj obrazy! 🎨

💡 **Przy kolejnych uruchomieniach**: Modele zostaną skopiowane z Google Drive (zajmie ~30 sekund zamiast 10 minut!)

---

### Features:
- 🚀 Automatyczny setup w Google Colab
- 💾 **Modele zapisywane w Google Drive** - pobierasz tylko raz!
- 🎨 Interaktywny interfejs Gradio
- 🔧 Konfigurowalne parametry (rozmiar, kroki, sampler, scheduler)
- 🌐 Publiczny URL do łatwego dostępu
- 🎲 Losowy lub stały seed
- ⚡ Szybkie ponowne uruchomienie (~2 min zamiast 15 min)


In [ ]:
# ============================================================
# GPU CHECK - This will stop if GPU is not enabled
# ============================================================
import torch
import sys

print("=" * 60)
print("🔍 CHECKING GPU AVAILABILITY...")
print("=" * 60)

if not torch.cuda.is_available():
    print("\n" + "🔴" * 30)
    print("❌ ERROR: NO GPU DETECTED!")
    print("🔴" * 30)
    print("\n⚠️  Google Colab requires MANUAL GPU activation.")
    print("\n📋 TO FIX THIS:")
    print("   1. Click: Runtime → Change runtime type")
    print("   2. Select: Hardware accelerator → GPU")
    print("   3. Click: Save")
    print("   4. Rerun this notebook (Runtime → Run all)")
    print("\n💡 SHORTCUT: Press Ctrl+M then G")
    print("\n" + "=" * 60)
    raise RuntimeError("GPU not enabled. Please enable GPU and restart.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"\n✅ GPU DETECTED!")
print(f"   📱 Device: {gpu_name}")
print(f"   💾 Memory: {gpu_memory:.1f} GB")
print("=" * 60)

# Mount Google Drive for persistent storage
# print("\n💾 Mounting Google Drive for persistent storage...")
# from google.colab import drive
# drive.mount('/content/drive')
# print("✅ Google Drive mounted!")

# Setup and install dependencies
print("\n📂 Setting up workspace...")
%cd /content
!git clone -b totoro3 https://github.com/camenduru/ComfyUI /content/TotoroUI
%cd /content/TotoroUI

print("\n📦 Installing dependencies (this will take a few minutes)...")
!apt -y install -qq aria2

# Install PyTorch and dependencies - use latest stable versions
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers torchsde einops diffusers accelerate gradio

print("\n✅ Dependencies installed!")


In [ ]:
# Download or copy models from Google Drive
import os

# Define paths
DRIVE_MODEL_PATH = "/content/drive/MyDrive/FLUX-models"
LOCAL_MODEL_PATHS = {
    "unet": "/content/TotoroUI/models/unet",
    "vae": "/content/TotoroUI/models/vae",
    "clip": "/content/TotoroUI/models/clip"
}

MODELS = {
    "flux1-dev-fp8.safetensors": ("unet", "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/flux1-dev-fp8.safetensors"),
    "ae.sft": ("vae", "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.sft"),
    "clip_l.safetensors": ("clip", "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors"),
    "t5xxl_fp8_e4m3fn.safetensors": ("clip", "https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors")
}

# Create Drive directory if it doesn't exist
# os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)

print("📥 Checking models...")
models_to_download = []

for model_name, (model_type, url) in MODELS.items():
    drive_model = f"{DRIVE_MODEL_PATH}/{model_name}"
    local_model = f"{LOCAL_MODEL_PATHS[model_type]}/{model_name}"
    
    # if os.path.exists(drive_model):
        # print(f"✅ Found in Drive: {model_name} - copying...")
        # !cp "{drive_model}" "{local_model}"
    # else:
        # print(f"❌ Not in Drive: {model_name} - will download")
    models_to_download.append((model_name, model_type, url, drive_model, local_model))

if models_to_download:
    print(f"\n📥 Downloading {len(models_to_download)} models (this will take 5-10 minutes, ~10GB)...")
    print("💡 Models will be saved to Google Drive for future use!")
    
    for model_name, model_type, url, drive_model, local_model in models_to_download:
        print(f"\n⬇️  Downloading {model_name}...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url} -d {LOCAL_MODEL_PATHS[model_type]} -o {model_name}
        
        # Copy to Drive for future use
        # print(f"💾 Saving to Drive...")
        # !cp "{local_model}" "{drive_model}"
    
    print("\n✅ All models downloaded and saved to Drive!")
else:
    print("\n✅ All models loaded from Drive (fast!)!")


In [ ]:
# Import libraries and load models
print("🔧 Loading models...")
import random
import torch
import numpy as np
from PIL import Image
import gradio as gr
import nodes
from nodes import NODE_CLASS_MAPPINGS
from totoro_extras import nodes_custom_sampler
from totoro import model_management

# Initialize node classes
DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
RandomNoise = nodes_custom_sampler.NODE_CLASS_MAPPINGS["RandomNoise"]()
BasicGuider = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicGuider"]()
KSamplerSelect = nodes_custom_sampler.NODE_CLASS_MAPPINGS["KSamplerSelect"]()
BasicScheduler = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicScheduler"]()
SamplerCustomAdvanced = nodes_custom_sampler.NODE_CLASS_MAPPINGS["SamplerCustomAdvanced"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

# Load models
with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")[0]
    unet = UNETLoader.load_unet("flux1-dev-fp8.safetensors", "fp8_e4m3fn")[0]
    vae = VAELoader.load_vae("ae.sft")[0]

print("✅ Models loaded successfully!")

# Helper functions
def closestNumber(n, m):
    """Round number to nearest multiple of m"""
    q = int(n / m)
    n1 = m * q
    if (n * m) > 0:
        n2 = m * (q + 1)
    else:
        n2 = m * (q - 1)
    if abs(n - n1) < abs(n - n2):
        return n1
    return n2

def generate_image(positive_prompt, width, height, seed, steps, sampler_name="euler", scheduler="simple"):
    """Generate image using FLUX.1-dev model"""
    try:
        with torch.inference_mode():
            # Handle random seed
            if seed == 0 or seed == -1:
                seed = random.randint(0, 18446744073709551615)
            print(f"🎲 Using seed: {seed}")

            # Encode prompt
            cond, pooled = clip.encode_from_tokens(clip.tokenize(positive_prompt), return_pooled=True)
            cond = [[cond, {"pooled_output": pooled}]]
            
            # Setup sampling
            noise = RandomNoise.get_noise(seed)[0] 
            guider = BasicGuider.get_guider(unet, cond)[0]
            sampler = KSamplerSelect.get_sampler(sampler_name)[0]
            sigmas = BasicScheduler.get_sigmas(unet, scheduler, steps, 1.0)[0]
            
            # Generate latent
            latent_image = EmptyLatentImage.generate(closestNumber(width, 16), closestNumber(height, 16))[0]
            
            # Sample
            print("🎨 Generating image...")
            sample, sample_denoised = SamplerCustomAdvanced.sample(noise, guider, sampler, sigmas, latent_image)
            
            # Decode
            model_management.soft_empty_cache()
            decoded = VAEDecode.decode(vae, sample)[0].detach()
            
            # Convert to PIL Image
            image_array = np.array(decoded*255, dtype=np.uint8)[0]
            result_image = Image.fromarray(image_array)
            
            print("✅ Image generated successfully!")
            return result_image, seed
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise e


In [ ]:
# Create and launch Gradio interface
with gr.Blocks(title="FLUX.1-dev Generator") as demo:
    gr.Markdown("# 🎨 FLUX.1-dev Image Generator")
    gr.Markdown("Generate high-quality images using FLUX.1-dev model")
    
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt",
                placeholder="Enter your image description...",
                value="black forest toast spelling out the words 'FLUX DEV', tasty, food photography, dynamic shot",
                lines=3
            )
            
            with gr.Row():
                width_input = gr.Slider(minimum=256, maximum=2048, value=1024, step=64, label="Width")
                height_input = gr.Slider(minimum=256, maximum=2048, value=1024, step=64, label="Height")
            
            with gr.Row():
                steps_input = gr.Slider(minimum=1, maximum=50, value=20, step=1, label="Steps")
                seed_input = gr.Number(label="Seed (0 for random)", value=0, precision=0)
            
            sampler_input = gr.Dropdown(
                choices=["euler", "euler_ancestral", "heun", "dpm_2", "dpm_2_ancestral", "lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_sde", "dpmpp_2m"],
                value="euler",
                label="Sampler"
            )
            
            scheduler_input = gr.Dropdown(
                choices=["simple", "normal", "karras", "exponential", "sgm_uniform"],
                value="simple",
                label="Scheduler"
            )
            
            generate_btn = gr.Button("🎨 Generate Image", variant="primary")
        
        with gr.Column():
            output_image = gr.Image(label="Generated Image", type="pil")
            used_seed = gr.Number(label="Used Seed", interactive=False)
    
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_input, width_input, height_input, seed_input, steps_input, sampler_input, scheduler_input],
        outputs=[output_image, used_seed]
    )

print("🚀 Starting Gradio interface...")
demo.launch(share=True, debug=True)
